# CMax V2 (`thesis_cmax_v2`) — one CMax gradient step per message-passing iteration drives R (walkthrough)

**What this unrolls**, in the order the harness runs it (`evaluation.py::experiment_tracking`, V2 branch):

| step | code |
|---|---|
| run config, frame-0 R from gyro | `run_config.py::RunConfig` |
| event frames `V` | `interacting_maps/data_loader.py::EventFrameSequence` |
| network build **with the CMax R-update** | `run_config.py::make_network` → `InteractingMapsThesis.enable_cmax_r_update` |
| raw undistorted events | `data_loader.py::load_events_fast`, `undistort_events` |
| per-frame warp inputs | `network_dissertation.py::Cost_CMax.set_frame` (called in `step`) |
| **analytic gradient** of the IWE variance | `angular_velocity.py::CMaxAngularVelocity._warp_and_jacobian`, `_contrast_and_grad` |
| **R ascent step** inside each iteration | `network_dissertation.py::Cost_CMax.compute_and_send_gradients` |
| message-passing loop | `InteractingMapsThesis.step(V, events=win)` |
| multi-frame loop + metrics | `evaluation.py::experiment_tracking`, `metrics.py::compute_metrics` |

Every unrolled block ends with an assertion against the real module function.
Inputs: `data/poster_rotation/` (seg_C). Outputs: figures + one tracking run under `experiments/cmax_walkthrough/`.

**In V2, R is driven only by CMax:** `Cost_Kinematics` stops updating R (it only pulls F toward C·R), there is
no `Cost_IMU`, and `Cost_CMax` adds `R ← R + lr·∂Var/∂R` once per iteration — `n_iters` small ascent steps per
frame on the **unblurred** IWE variance, warm-started from the previous frame's R (the network is recurrent).
The rest of the network (I, G, F) never feeds back into R.

In [ ]:
# setup — run from the repo root (data/ and experiments/ are relative paths)
import os, sys
os.environ.setdefault('IM_MLFLOW', '0')          # walkthrough runs never go to MLflow
ROOT = os.path.abspath('.')
assert os.path.exists(os.path.join(ROOT, 'config.py')), 'start the notebook from the repo root'
sys.path.insert(0, ROOT)
# local slice switches (env): CMAX_WT_NFRAMES (default 5), CMAX_WT_NITERS (default ITERS_PER_FRAME), CMAX_WT_K (frame studied in detail)

In [ ]:
# imports — only what the cells use, grouped by module
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from config import ITERS_PER_FRAME
from run_config import RunConfig, make_network
from interacting_maps.data_loader import (EventFrameSequence, CameraCalibration,
                                          load_events_fast, undistort_events)
from interacting_maps.eval_io import (load_imu, load_groundtruth, load_omega_gt,
                                      get_gyro_for_frame, get_reference_omega)
from interacting_maps.metrics import compute_metrics
from interacting_maps.angular_velocity import CMaxAngularVelocity
from interacting_maps.network_dissertation import InteractingMapsThesis
from interacting_maps.camera import CLIP_I, CLIP_G, CLIP_F, CLIP_R
from interacting_maps.style import use_style, save, to_deg_s, OMEGA_AXIS_COLORS, SCREEN_WIDTH
from interacting_maps.viz import draw_map, map_scale
import evaluation as E   # only for the final end-to-end check (experiment_tracking)

In [ ]:
# constants — the walkthrough's knobs (mirror tests/test_cmax_*.py: poster seg_C, ω_x-dominant, |ω|≈1.1 rad/s)
DATASET    = 'poster_rotation'
SEGMENT_ID = 'seg_C'                                         # config.DATASET_SEGMENTS
MODEL      = 'thesis_cmax_v2'
N_FRAMES   = int(os.environ.get('CMAX_WT_NFRAMES', 5))       # segment default is 150
N_ITERS    = int(os.environ.get('CMAX_WT_NITERS', ITERS_PER_FRAME))   # config.ITERS_PER_FRAME = 75
K          = int(os.environ.get('CMAX_WT_K', 0))             # frame studied step by step
OUT_ROOT   = 'experiments/cmax_walkthrough'                  # NOT results/ (load-bearing)
FIG_DIR    = os.path.join(OUT_ROOT, 'v2_figures')
os.makedirs(FIG_DIR, exist_ok=True)
np.set_printoptions(precision=4, suppress=True)

## 1) Run configuration — `run_config.py::RunConfig`
`RunConfig(model='thesis_cmax_v2')` decides which costs are active and where the frame-0 `R` comes from.
For V2: `use_imu=False` (no anchor), `use_cmax=True`, `use_cmax_v2=True`, and `cmax_lr` = the ascent
step (default `1e-4`; cmax.md: stable ~[1e-5, 1e-4], diverges ≳5e-4).
`initial_R` is computed in `RunConfig._compute_initial_R`: the **gyro** mean over the first frame window,
times `dt` (rad/frame). This is the only place the IMU enters a CMax run.

In [ ]:
# run_config.py::RunConfig.__init__
rc = RunConfig(dataset=DATASET, model=MODEL, segment=SEGMENT_ID,
               n_frames=N_FRAMES, n_iters=N_ITERS, out_root=OUT_ROOT)
rc

In [ ]:
# the switches this model sets, the deltas, and the frame-0 R (rad/frame and °/s)
print({k: getattr(rc, k) for k in ('use_thesis', 'use_imu', 'use_cmax', 'use_cmax_v2',
                                   'cmax_lr', 'distortion_mode', 'poisson', 'frame_duration')})
print('params     :', rc.params)
print('initial_R  :', rc.initial_R, 'rad/frame  =', to_deg_s(rc.initial_R / rc.frame_duration), '°/s')
print('output_dir :', rc.output_dir)

## 2) Event frames `V` — `interacting_maps/data_loader.py::EventFrameSequence`
The network's input. Events are read once for `[t_start, t_start + n·dt + 0.1 s)`; each frame bins its window
into a signed count image, clipped at ±10 and divided by 10 (`clip_value=10.0` in `experiment_tracking`).
In `C_full` mode the events are **not** undistorted here (lens handled inside the kinematic matrix `C`).

In [ ]:
# evaluation.py::experiment_tracking — EventFrameSequence(...) exactly as the harness builds it
seq = EventFrameSequence(
    rc.paths['events'], rc.paths['calib'],
    frame_duration=rc.frame_duration, t_start=rc.t_start,
    n_frames=rc.n_frames, clip_value=10.0,
    undistort=rc.undistort_at_event_level,
    sensor_size=rc.sensor_size,
)
H, W = seq.H, seq.W
fx, fy, cx, cy = seq.calib.fx, seq.calib.fy, seq.calib.cx, seq.calib.cy
frames = list(seq)                         # [(V, t_mid), ...]
print('undistort at event level:', seq.undistort, '| dist coeffs:', seq.calib.dist)

In [ ]:
# what V looks like: per frame, events binned, share of saturated pixels (|V| = 1 ↔ ≥10 net events)
pd.DataFrame([{'frame': k, 't_mid': t, 'nonzero_px': int((V != 0).sum()),
               'saturated_px': int((np.abs(V) >= 1).sum()), 'V_min': V.min(), 'V_max': V.max()}
              for k, (V, t) in enumerate(frames)])

In [ ]:
# sensors used to SCORE (Vicon, independent) and the gyro (frame-0 init + cross-check only)
imu_data = load_imu(rc.paths['imu'])
gt_data  = load_groundtruth(rc.paths['groundtruth'])
om_data  = load_omega_gt(rc.paths.get('omega_gt'))
print('imu rows', len(imu_data), '| gt rows', len(gt_data), '| omega_gt', om_data is not None)

## 3) Network — `run_config.py::make_network` (thesis branch + V2 switch, unrolled)
After the normal build, `make_network` creates a `CMaxAngularVelocity` (pinhole, `use_polarity=True`) and calls
`enable_cmax_r_update(est, lr)`: `r_update='cmax'`, `cost_kin.update_r=False`, and a `Cost_CMax` is appended to
`net.costs`. Only then `initialize_from_rotation(initial_R)` (gyro).

In [ ]:
# run_config.py::make_network — dist_coeffs for C_full, then the thesis network
dist_coeffs = CameraCalibration(rc.paths['calib']).dist if rc.distortion_mode == 'C_full' else None
net = InteractingMapsThesis(H=H, W=W, fx=fx, fy=fy, cx=cx, cy=cy,
                            frame_duration=rc.frame_duration, dist_coeffs=dist_coeffs,
                            poisson=rc.poisson, **rc.params)
print('before V2 switch:', [type(c).__name__ for c in net.costs], '| kinematics updates R:', net.cost_kin.update_r)

In [ ]:
# run_config.py::make_network — V2: estimator + InteractingMapsThesis.enable_cmax_r_update
est = CMaxAngularVelocity(H, W, fx, fy, cx, cy, use_polarity=True)
net.enable_cmax_r_update(est, lr=rc.cmax_lr)
print('after V2 switch :', [type(c).__name__ for c in net.costs], '| kinematics updates R:', net.cost_kin.update_r,
      '| r_update:', net.r_update)
print('Cost_CMax: lr', net.cost_cmax.lr, '| dt_frame', net.cost_cmax.dt_frame,
      '| estimator blur_sigma', est.blur_sigma, '(unused by the gradient path) | dist', est.dist)

In [ ]:
# run_config.py::make_network — initialize_from_rotation(initial_R): R = gyro·dt, F = C·R, I = noise
net.initialize_from_rotation(rc.initial_R)
print('|F - C·R| max:', np.abs(net.F - np.einsum('hwij,j->hwi', net._C_mat, net.R)).max(),
      '| R °/s:', to_deg_s(net.R / rc.frame_duration))

In [ ]:
# check: make_network == unrolled build
net_chk = make_network(rc, H, W, fx, fy, cx, cy)
for q in ('q_I', 'q_G', 'q_F', 'q_R'):
    assert np.array_equal(getattr(net, q).value, getattr(net_chk, q).value), q
assert [type(c) for c in net.costs] == [type(c) for c in net_chk.costs]
assert net_chk.cost_kin.update_r is False and net_chk.cost_cmax.lr == rc.cmax_lr
print('[ok] make_network == unrolled build')

## 4) Events for CMax — `load_events_fast` + `undistort_events` (`evaluation.py::experiment_tracking`, "B1 plumbing")
CMax does not see `V`; it gets the **raw event list**, re-loaded over the same span and **undistorted** with
`cv2.undistortPoints` (pixel coordinates become floats, same `K`). So in `C_full` mode the network works on
distorted pixels with a distortion-aware `C`, while CMax works on undistorted pixels with a pure pinhole warp —
two representations of the same geometry. Points that undistort outside the sensor are kept here; they are only
dropped later when they are voted into the IWE.

In [ ]:
# evaluation.py::experiment_tracking — dur = n_frames·dt + 0.1, then undistort ONCE
dur = rc.n_frames * rc.frame_duration + 0.1
raw_events = load_events_fast(rc.paths['events'], t_start=rc.t_start, duration=dur)
cmax_events = undistort_events(raw_events, seq.calib)
print('events:', len(cmax_events), '| same span as EventFrameSequence:',
      np.array_equal(raw_events, seq._events))

In [ ]:
# what undistortion does to the coordinates: shift size and how many leave the sensor
shift = np.hypot(cmax_events[:, 1] - raw_events[:, 1], cmax_events[:, 2] - raw_events[:, 2])
outside = ((cmax_events[:, 1] < 0) | (cmax_events[:, 1] > W - 1) |
           (cmax_events[:, 2] < 0) | (cmax_events[:, 2] > H - 1))
print(f'shift px: median {np.median(shift):.2f}, p99 {np.percentile(shift, 99):.2f}, max {shift.max():.2f}')
print(f'outside the sensor after undistortion: {outside.mean():.2%}')
print('polarity values:', np.unique(cmax_events[:, 3]))

## 5) Frame window `K` — `evaluation.py::experiment_tracking` (per-frame slice)
The harness slices `[t_lo, t_hi)` out of the undistorted list for each frame. Below everything is done for one
frame `K` (default 0). `step` passes `t_ref` = midpoint of the **event** timestamps (not the frame centre V1 uses).

In [ ]:
# evaluation.py::experiment_tracking — window bounds, slice, gyro (input) and reference (scoring)
t_lo = rc.t_start + K * rc.frame_duration
t_hi = t_lo + rc.frame_duration
V, t_mid = frames[K]
win = cmax_events[(cmax_events[:, 0] >= t_lo) & (cmax_events[:, 0] < t_hi)]
omega_imu = get_gyro_for_frame(imu_data, t_lo, t_hi)
omega_ref, ref_src = get_reference_omega(gt_data, imu_data, t_lo, t_hi, om_data)
print(f'frame {K}: [{t_lo:.4f}, {t_hi:.4f})  events {len(win)}  ON share {win[:, 3].mean():.2%}')
print('ω gyro  °/s :', to_deg_s(omega_imu))
print(f'ω ref   °/s : {to_deg_s(omega_ref)}   ({ref_src})')

In [ ]:
# the two candidate reference times: frame centre (V1 passes this) vs event-span midpoint (V2 / estimate default)
t_ref_frame  = 0.5 * (t_lo + t_hi)
t_ref_events = 0.5 * (win[:, 0].min() + win[:, 0].max())
print(f'frame centre {t_ref_frame:.6f} s | event midpoint {t_ref_events:.6f} s | '
      f'difference {(t_ref_events - t_ref_frame) * 1e3:+.3f} ms')

## 6) Per-frame warp inputs — `Cost_CMax.set_frame` (called at the top of `step`)
Computed once per frame and cached on the cost: unit bearings, `dt = t − t_ref`, signed weights. Every one of the
`n_iters` gradient evaluations reuses them; only ω changes.

In [ ]:
# keep an untouched copy for the step() equality check
net_ref = copy.deepcopy(net)
cc = net.cost_cmax

In [ ]:
# network_dissertation.py::step + Cost_CMax.set_frame — unrolled
net.q_V.value = V
net._use_imu = False                                       # omega_imu=None → Cost_IMU skipped
t_ref = 0.5 * (win[:, 0].min() + win[:, 0].max())          # step(): event-span midpoint
assert len(win) >= 10                                      # set_frame(): guard
xs = win[:, 1].astype(np.float64)
ys = win[:, 2].astype(np.float64)
cc._bearings = cc.est._bearings(xs, ys)
cc._dt = win[:, 0].astype(np.float64) - t_ref
pol = win[:, 3].astype(np.float64)
cc._w = (2.0 * pol - 1.0) if cc.est.use_polarity else np.ones_like(pol)
print('events', len(cc._w), '| dt range ms', cc._dt.min() * 1e3, cc._dt.max() * 1e3, '| Σw', cc._w.sum())

## 7) One gradient evaluation — `CMaxAngularVelocity._contrast_and_grad` (unrolled)
`Cost_CMax` converts the network's `R` (rad/frame) to `ω = R/dt_frame` (rad/s) and asks for the value and
gradient of the **unblurred** IWE variance. Note this objective differs from V1 (which maximizes the
*blurred* variance with Nelder-Mead) — the two methods do not share an optimum exactly.
### 7a) Warp + per-event Jacobian — `_warp_and_jacobian`
`J_e = ∂(x', y')/∂ω` (N×2×3, px per rad/s): how far each event's warped position moves per unit of ω.

In [ ]:
# network_dissertation.py::Cost_CMax.compute_and_send_gradients — rad/frame → rad/s
omega = net.R / cc.dt_frame
print('ω at the start of the frame °/s:', to_deg_s(omega))

In [ ]:
# angular_velocity.py::_warp_and_jacobian
xw, yw, J = est._warp_and_jacobian(cc._bearings, cc._dt, omega)
xw_chk, yw_chk = est._warp_to_pixels(cc._bearings, cc._dt, omega)
print('warp agrees with _warp_to_pixels:', np.allclose(xw, xw_chk) and np.allclose(yw, yw_chk))
Jn = np.linalg.norm(J, axis=1)                             # (N,3): px per rad/s, per axis
print('median |∂x\'/∂ω_j| px per rad/s (x, y, z):', np.median(Jn, axis=0))

### 7b) Bilinear voting of the IWE and of its 3 derivative images `D_j = ∂IWE/∂ω_j`
Same 4-corner bilinear kernel as V1's IWE; the derivative images use the kernel's derivative times `J`.
Votes outside the sensor are dropped from both.

In [ ]:
# angular_velocity.py::_contrast_and_grad — corner loop
iwe = np.zeros((H, W)); D = np.zeros((3, H, W))
x0 = np.floor(xw).astype(np.int64); y0 = np.floor(yw).astype(np.int64)
dx = xw - x0; dy = yw - y0
Jx, Jy = J[:, 0, :], J[:, 1, :]
corners = [
    (0, 0, (1 - dx) * (1 - dy), -(1 - dy), -(1 - dx)),
    (1, 0, dx * (1 - dy),        (1 - dy), -dx),
    (0, 1, (1 - dx) * dy,       -dy,        (1 - dx)),
    (1, 1, dx * dy,              dy,         dx),
]
for ox, oy, kw, dkx, dky in corners:
    xi = x0 + ox; yi = y0 + oy
    m = (xi >= 0) & (xi < W) & (yi >= 0) & (yi < H)
    ym, xm = yi[m], xi[m]
    np.add.at(iwe, (ym, xm), (cc._w * kw)[m])
    dKdw = dkx[:, None] * Jx + dky[:, None] * Jy
    contrib = (cc._w[:, None] * dKdw)[m]
    for j in range(3):
        np.add.at(D[j], (ym, xm), contrib[:, j])
print('ΣIWE', iwe.sum(), 'vs Σw', cc._w.sum(), '| ΣD_j', D.sum(axis=(1, 2)))

### 7c) Value and gradient
`dVar/dω_j = (2/P)·Σ_u IWE_u·D_j(u)`. The exact derivative is `(2/P)·Σ_u (IWE_u − mean)·D_j(u)`; the code drops
the `mean·ΣD_j` term, justified by "voting mass is conserved" (`ΣD_j = 0`). That holds only if no vote crosses the
sensor border — the cell below measures the size of the dropped term.

In [ ]:
# angular_velocity.py::_contrast_and_grad — var and grad
var = float(np.var(iwe))
P = iwe.size
grad = np.array([(2.0 / P) * np.sum(iwe * D[j]) for j in range(3)])
grad_exact = np.array([(2.0 / P) * np.sum((iwe - iwe.mean()) * D[j]) for j in range(3)])
print('var', var, '| grad (code)', grad, '| grad (with mean term)', grad_exact)
print('relative size of the dropped term:', np.linalg.norm(grad - grad_exact) / np.linalg.norm(grad_exact))

In [ ]:
# check: _contrast_and_grad == unrolled; and the gradient vs central finite differences of Var(IWE)
v_chk, g_chk = est._contrast_and_grad(omega, cc._bearings, cc._dt, cc._w)
assert v_chk == var and np.array_equal(g_chk, grad)
h = 1e-3
fd = np.array([(np.var(est._build_iwe(cc._bearings, cc._dt, cc._w, omega + h * e)) -
                np.var(est._build_iwe(cc._bearings, cc._dt, cc._w, omega - h * e))) / (2 * h) for e in np.eye(3)])
print('[ok] _contrast_and_grad == unrolled | FD grad', fd, '| rel. diff', np.linalg.norm(fd - grad) / np.linalg.norm(fd))

### 7d) From gradient to an R message — `Cost_CMax.compute_and_send_gradients`
Chain rule `∂Var/∂R = (∂Var/∂ω)/dt_frame`; ascent `R += lr·∂Var/∂R`, sent as accumulator `−lr·∂Var/∂R` because
Phase 2 does `value −= accumulator`. In ω units one iteration moves ω by `lr·grad/dt_frame²` — the cell shows it.

In [ ]:
# network_dissertation.py::Cost_CMax.compute_and_send_gradients — the message on R
grad_R = grad / cc.dt_frame
msg_R = -cc.lr * grad_R
d_omega = -msg_R / cc.dt_frame
print('accumulator on R (rad/frame):', msg_R)
print('ω change from ONE iteration °/s:', to_deg_s(d_omega), '| ×', rc.n_iters, 'iters if grad stayed constant:',
      to_deg_s(d_omega * rc.n_iters))

## 8) The message-passing loop — `InteractingMapsThesis.step(V, events=win)` (unrolled)
Each iteration: Phase 1 (all costs; `Cost_Kinematics` now sends nothing to R, `Cost_IMU` skipped), Phase 2
update, clip. Recorded per iteration: R, the CMax message, the unblurred contrast at the new R
(`Cost_CMax.build_current_iwe`). If the contrast is still rising at the last iteration, `n_iters·lr` is too small
to reach the CMax optimum within a frame (it carries over to the next frame through the recurrent R).

In [ ]:
# network_dissertation.py::step — n_iters loop, recording R, R messages and contrast
R_iters, R_msgs, contrast_iters = [net.q_R.value.copy()], [], [cc.build_current_iwe()[1]]
for _ in range(rc.n_iters):
    # PHASE 1
    for q in [net.q_I, net.q_G, net.q_F, net.q_R]:
        q.reset_gradient()
    msgs = {}
    for cost in net.costs:
        if cost is net.cost_imu and not net._use_imu:
            continue
        before = net.q_R.gradient_accumulator.copy()
        cost.compute_and_send_gradients()
        msgs[type(cost).__name__] = net.q_R.gradient_accumulator - before
    R_msgs.append(msgs)
    # PHASE 2
    net.q_I.update(1.0); net.q_G.update(1.0); net.q_F.update(1.0); net.q_R.update(1.0)
    net.q_I.value = np.clip(net.q_I.value, -CLIP_I, CLIP_I)
    net.q_G.value = np.clip(net.q_G.value, -CLIP_G, CLIP_G)
    net.q_F.value = np.clip(net.q_F.value, -CLIP_F, CLIP_F)
    net.q_R.value = np.clip(net.q_R.value, -CLIP_R, CLIP_R)
    R_iters.append(net.q_R.value.copy())
    contrast_iters.append(cc.build_current_iwe()[1])
R_iters = np.array(R_iters)
print('R after step °/s:', to_deg_s(net.R / rc.frame_duration))
print('costs that sent a message to R:', sorted({k for m in R_msgs for k, v in m.items() if np.any(v)}))

In [ ]:
# check: InteractingMapsThesis.step == unrolled loop
net_ref.step(V, n_iters=rc.n_iters, events=win)
for q in ('q_I', 'q_G', 'q_F', 'q_R'):
    assert np.array_equal(getattr(net, q).value, getattr(net_ref, q).value), q
print('[ok] step() == unrolled')

### 8b) Where would CMax have converged? — reference optima on the same window
From the same start ω: the **unblurred** optimum (`use_analytic_gradient=True` → CG, the objective V2 ascends)
and the **blurred** Nelder-Mead optimum (what V1 would anchor to). Inspection only.

In [ ]:
# angular_velocity.py::estimate — reference optima from the frame's starting ω (same t_ref as V2)
omega_start = R_iters[0] / rc.frame_duration
est_cg = CMaxAngularVelocity(H, W, fx, fy, cx, cy, use_polarity=True, use_analytic_gradient=True)
omega_cg = est_cg.estimate(win, t_ref=t_ref, omega_init=omega_start)
omega_nm = CMaxAngularVelocity(H, W, fx, fy, cx, cy, use_polarity=True).estimate(win, t_ref=t_ref, omega_init=omega_start)
omega_v2 = net.R / rc.frame_duration
rows_opt = []
for name, w in (('start (gyro init)', omega_start), ('V2 after step', omega_v2),
                ('CG unblurred opt', omega_cg), ('NM blurred opt (V1)', omega_nm), ('gyro', omega_imu)):
    rows_opt.append({'ω': name, **{f'w{c} °/s': v for c, v in zip('xyz', to_deg_s(w))},
                     'unblurred var': np.var(est._build_iwe(cc._bearings, cc._dt, cc._w, w)),
                     'err vs ref °/s': compute_metrics(w, omega_ref)[0]})
pd.DataFrame(rows_opt).round(4)

In [ ]:
# R and contrast over the iterations
it = np.arange(rc.n_iters + 1)
with use_style('screen'):
    fig, axes = plt.subplots(1, 4, figsize=(SCREEN_WIDTH, 3))
    for a in range(3):
        ax = axes[a]
        ax.plot(it, to_deg_s(R_iters[:, a] / rc.frame_duration), color=OMEGA_AXIS_COLORS[a], label='network R')
        ax.axhline(to_deg_s(omega_cg[a]), color='m', lw=0.9, ls='--', label='CG optimum')
        ax.axhline(to_deg_s(omega_ref[a]), color='k', lw=1.2, label='reference')
        ax.axhline(to_deg_s(omega_imu[a]), color='0.55', ls=':', label='gyro')
        ax.set_title(f'$\\omega_{"xyz"[a]}$ (°/s)'); ax.set_xlabel('iteration')
    axes[0].legend(fontsize=7)
    axes[3].plot(it, contrast_iters, color='0.25'); axes[3].set_title('unblurred IWE variance'); axes[3].set_xlabel('iteration')
    path = os.path.join(FIG_DIR, 'v2_R_over_iterations.png'); save(fig, path)
display(Image(path))

In [ ]:
# IWE at the frame's start ω, after the V2 step, and at the CG optimum (shared scale)
iwe_start = est._build_iwe(cc._bearings, cc._dt, cc._w, omega_start)
iwe_v2, _ = cc.build_current_iwe()
iwe_cg = est._build_iwe(cc._bearings, cc._dt, cc._w, omega_cg)
s = map_scale('IWE', iwe_cg)
with use_style('screen'):
    fig, axes = plt.subplots(1, 4, figsize=(SCREEN_WIDTH, 2.6))
    draw_map(axes[0], 'V', V, title=f'V (frame {K})')
    draw_map(axes[1], 'IWE', iwe_start, title='IWE at start ω', scale=s)
    draw_map(axes[2], 'IWE', iwe_v2, title='IWE after V2 step', scale=s)
    draw_map(axes[3], 'IWE', iwe_cg, title='IWE at CG optimum', scale=s)
    path = os.path.join(FIG_DIR, 'v2_iwe.png'); save(fig, path)
display(Image(path))

## 9) All frames — `evaluation.py::experiment_tracking` V2 branch (unrolled)
Fresh network, then per frame: slice → `step(V, events=win)` → `ω_est = R/dt` → `compute_metrics`.
R carries over between frames (warm start = previous frame's end state). `contrast` is the unblurred variance at
the final R of the frame (what `--save-iwe` logs).

In [ ]:
# evaluation.py::experiment_tracking — V2 loop
net = make_network(rc, H, W, fx, fy, cx, cy)
rows = []
for k, (V_k, t_mid_k) in enumerate(frames):
    t_lo_k = rc.t_start + k * rc.frame_duration
    t_hi_k = t_lo_k + rc.frame_duration
    omega_imu_k = get_gyro_for_frame(imu_data, t_lo_k, t_hi_k)
    omega_ref_k, _ = get_reference_omega(gt_data, imu_data, t_lo_k, t_hi_k, om_data)
    win_k = cmax_events[(cmax_events[:, 0] >= t_lo_k) & (cmax_events[:, 0] < t_hi_k)]
    omega_before = net.R / rc.frame_duration
    net.step(V_k, n_iters=rc.n_iters, events=win_k)
    omega_est = net.R / rc.frame_duration
    err, dir_err, beta = compute_metrics(omega_est, omega_ref_k)
    rows.append({'frame': k, 't_mid': t_mid_k, 'n_ev': len(win_k),
                 **{f'est_w{c}': v for c, v in zip('xyz', to_deg_s(omega_est))},
                 **{f'ref_w{c}': v for c, v in zip('xyz', to_deg_s(omega_ref_k))},
                 **{f'imu_w{c}': v for c, v in zip('xyz', to_deg_s(omega_imu_k))},
                 'moved_in_frame': to_deg_s(np.linalg.norm(omega_est - omega_before)),
                 'contrast': net.cost_cmax.build_current_iwe()[1],
                 'err_est': err, 'err_gyro': compute_metrics(omega_imu_k, omega_ref_k)[0],
                 'dir_err': dir_err, 'beta': beta})
df = pd.DataFrame(rows)
df.round(3)

In [ ]:
# summary (°/s): V2 vs the gyro, both against the reference
df[['err_est', 'err_gyro', 'moved_in_frame', 'dir_err', 'beta']].describe().round(2)

In [ ]:
# check: evaluation.experiment_tracking (V2) == unrolled loop  (writes to experiments/cmax_walkthrough/)
E.experiment_tracking(rc, save_frames=False)
trk = pd.read_csv(os.path.join(rc.output_dir, 'tracking.csv'))
assert np.allclose(np.degrees(trk[['est_wx', 'est_wy', 'est_wz']].values),
                   df[['est_wx', 'est_wy', 'est_wz']].values, atol=1e-9)
assert np.allclose(trk['err_deg_s'].values, df['err_est'].values, atol=1e-9)
print('[ok] experiment_tracking == unrolled V2 loop')

In [ ]:
# ω per axis over the frames: estimate vs reference vs gyro (°/s)
t_axis = df['t_mid'] - rc.t_start
with use_style('screen'):
    fig, axes = plt.subplots(1, 3, figsize=(SCREEN_WIDTH, 3.2), sharex=True)
    for a, ax in enumerate(axes):
        ax.plot(t_axis, df[f'est_w{"xyz"[a]}'], color=OMEGA_AXIS_COLORS[a], label='V2 network ω')
        ax.plot(t_axis, df[f'ref_w{"xyz"[a]}'], color='k', lw=1.2, label='reference')
        ax.plot(t_axis, df[f'imu_w{"xyz"[a]}'], color='0.55', ls=':', label='gyro')
        ax.set_title(f'$\\omega_{"xyz"[a]}$ (°/s)')
        ax.set_xlabel('time (s)')
    axes[0].legend(fontsize=7)
    path = os.path.join(FIG_DIR, 'v2_omega_over_frames.png')
    save(fig, path)
display(Image(path))

## What to check
- **3 init:** frame 0 starts from the gyro (`initial_R`) — so V2 is IMU-free only from frame 1 on.
- **7c dropped mean term:** is the relative size negligible on this window? It grows when many votes cross the border.
- **7d step size:** ω change per iteration and per frame vs the distance to the optimum; with fixed `lr` this
  scales with (event count)², so it is data-dependent.
- **8 / 8b convergence:** is the contrast still rising at the last iteration? How far is "V2 after step" from the
  CG optimum? Is the CG (unblurred) optimum close to the NM (blurred, V1) optimum?
- **8 isolation:** only `Cost_CMax` sends messages to R — so in V2 the maps I, G, F have **no** influence on ω;
  the network is a CMax tracker plus maps that follow it (`F ← C·R`).
- **9 `moved_in_frame`:** small values with large errors mean R is stuck at its warm start (step too small).